In [ ]:
# import
from transformers import AutoProcessor, AutoModel
from PIL import Image
import torch

# load model
device = "cuda"
processor_name_or_path = "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"
model_pretrained_name_or_path = "yuvalkirstain/PickScore_v1"

processor = AutoProcessor.from_pretrained(processor_name_or_path)
model = AutoModel.from_pretrained(model_pretrained_name_or_path).eval().to(device)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("weathon/aas_benchmark_final")

In [ ]:

def calc_probs(prompt, images):
    image_inputs = processor(
        images=images,
        padding=True,
        truncation=True,
        max_length=77,
        return_tensors="pt",
    ).to(device)

    text_inputs = processor(
        text=prompt,
        padding=True,
        truncation=True,
        max_length=77,
        return_tensors="pt",
    ).to(device)


    with torch.no_grad():
        # embed
        image_embs = model.get_image_features(**image_inputs)
        image_embs = image_embs / torch.norm(image_embs, dim=-1, keepdim=True)

        text_embs = model.get_text_features(**text_inputs)
        text_embs = text_embs / torch.norm(text_embs, dim=-1, keepdim=True)

        # score
        scores = model.logit_scale.exp() * (text_embs @ image_embs.T)[0]

        # get probabilities if you have multiple images to choose from
        probs = torch.softmax(scores, dim=-1)

    return probs.cpu().tolist()

In [ ]:
dataset = dataset["train"]
dataset

Dataset({
    features: ['image_original', 'image_distorted', 'index', 'prompt_original', 'prompt_distorted', 'selected_dims', 'llm_judge', 'hpsv2_reward', 'llm_selected', 'blip_selected', 'model', 'blip_score', 'image_reward', 'hpsv3_reward', 'rater', 'blip_score_original'],
    num_rows: 3300
})

In [ ]:
def rate_pick_score(sample):
  return calc_probs(sample["prompt_distorted"], [sample["image_original"], sample["image_distorted"]])

In [ ]:
scores = []
for i in range(len(dataset)):
  scores.append(rate_pick_score(dataset[i]))

In [ ]:
dataset = dataset.add_column("pick_scores", scores)

In [ ]:
dataset.push_to_hub("weathon/aas_benchmark_final")

In [ ]:
import numpy as np
gt = dataset["llm_selected"]
pred = np.array(scores)[:,1]
valid_idx = np.array(gt) != -1
gt = np.array(gt)[valid_idx]
pred = pred[valid_idx]

In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(gt, pred)

np.float64(0.7131505217055261)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(gt, np.round(pred))

0.850845948352627

np.float64(0.6071106530980291)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score

from sklearn.metrics import f1_score
accuracy_score(gt, np.round(pred)), balanced_accuracy_score(gt, np.round(pred)), f1_score(gt, np.round(pred))

(0.850845948352627, np.float64(0.6071106530980291), 0.9190234469422287)

In [ ]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
from datasets import load_dataset
dataset = load_dataset("weathon/aas_benchmark_final")

In [ ]:
dataset = dataset["train"]

In [ ]:
model = model.cuda()

In [ ]:
import torch
def score_clip(sample):
  inputs = processor(text=sample["prompt_distorted"], images=[sample["image_original"], sample["image_distorted"]], return_tensors="pt", padding=True, truncation=True).to("cuda")
  with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image.flatten()
    probs = logits_per_image.softmax(dim=0)
    return probs.cpu().tolist()


In [ ]:
import tqdm
clip_scores = []
for sample in tqdm.tqdm(dataset):
  clip_scores.append(score_clip(sample))

100%|██████████| 3300/3300 [10:27<00:00,  5.26it/s]


In [ ]:
dataset = dataset.add_column("clip_scores", clip_scores)

In [ ]:
dataset.push_to_hub("weathon/aas_benchmark_final")

Uploading the dataset shards:   0%|          | 0/15 [00:00<?, ? shards/s]

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   5%|5         | 41.9MB /  778MB            

CommitInfo(commit_url='https://huggingface.co/datasets/weathon/aas_benchmark_final/commit/2bd9d0c8e8f89631d27230cef2069cae24eda60b', commit_message='Upload dataset', commit_description='', oid='2bd9d0c8e8f89631d27230cef2069cae24eda60b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/weathon/aas_benchmark_final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='weathon/aas_benchmark_final'), pr_revision=None, pr_num=None)

In [ ]:
import numpy as np
gt = dataset["llm_selected"]
pred = np.array(clip_scores)[:,1]
valid_idx = np.array(gt) != -1
gt = np.array(gt)[valid_idx]
pred = pred[valid_idx]

In [ ]:
from sklearn.metrics import roc_auc_score


np.float64(0.8104711451758341)

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
accuracy_score(gt, np.round(pred)), balanced_accuracy_score(gt, np.round(pred)), f1_score(gt, np.round(pred)), roc_auc_score(gt, pred)

(0.9127337488869101, np.float64(0.6560769032590493), 0.9541413196069256)